# Challenge 3: Datasets, versions and splits

**EDD: Define. Model upgrade framework, step 3: build and version the golden dataset.** A dataset is the part of an eval you control. You'll build one from production traces,
build one directly in code, tag a version, use splits (`dev`-style `train` to tune against, `test` for the gate), and move a dataset out of LangSmith and back in.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from eval_lab.data import EXAMPLES, EXTRA_EXAMPLES
TRACES_DS = lab_sdk.ds_traces()
UPLOADED = lab_sdk.ds_uploaded()

## 1. A dataset from traces

Good production runs become regression cases. This cell takes successful v1.1 runs, redacts the customer text,
and stores each run's id in the example's `metadata` so you can trace an example back to its source.

In [ ]:
good = [r for r in lab_sdk.roots(client)
        if lab_sdk.run_meta(r).get("release") == "v1.1" and not r.error and "thread" not in (r.tags or [])]

if client.has_dataset(dataset_name=TRACES_DS):
    print(f"'{TRACES_DS}' already exists; leaving it as is")
else:
    ds = client.create_dataset(TRACES_DS, description="Regression cases captured from v1.1 production traces")
    client.create_examples(dataset_id=ds.id, examples=[
        {
            "inputs": {"query": lab_sdk.redact(r.inputs["query"])},
            "outputs": {"answer": lab_sdk.redact(r.outputs["answer"]), "trajectory": r.outputs["trajectory"]},
            "metadata": {"source_run_id": str(r.id), "release": "v1.1"},
        }
        for r in good
    ])
    print(f"Created '{TRACES_DS}' with {len(good)} examples")

## 2. A dataset in code

The golden set is hand-written and versioned with your code. `EXAMPLES` has fifteen cases; each carries the
expected tool trajectory and keywords, a `segment` in its metadata, and a `split` (`train` or `test`). Challenges 5 and 8 run on the `test` split, so it is never tuned against.

In [ ]:
for e in EXAMPLES:
    print(f"{e['split']:5} {e['metadata']['segment']:13} {e['inputs']['query']}")

Create the dataset with `client.create_dataset()` and `client.create_examples()`. A reference solution is in the next cell.

In [ ]:
# Reference solution: try writing this yourself first.
if client.has_dataset(dataset_name=GOLDEN):
    golden = client.read_dataset(dataset_name=GOLDEN)
    print(f"'{GOLDEN}' already exists; leaving it as is")
else:
    golden = client.create_dataset(GOLDEN, description="Golden order-support cases")
    client.create_examples(dataset_id=golden.id, examples=EXAMPLES)
    print(f"Created '{GOLDEN}' with {len(EXAMPLES)} examples")

## 3. Versions and tags

Every change to a dataset creates a new **version**. Tag the one you evaluated against, so an experiment can
say exactly which data it saw. This cell tags the current version `v1-baseline`, then adds two more examples.

In [ ]:
import time

if len(list(client.list_examples(dataset_name=GOLDEN))) < len(EXAMPLES) + len(EXTRA_EXAMPLES):
    time.sleep(3)
    base = max(client.list_dataset_versions(dataset_name=GOLDEN), key=lambda v: v.as_of)
    client.update_dataset_tag(dataset_name=GOLDEN, as_of=base.as_of, tag="v1-baseline")
    client.create_examples(dataset_name=GOLDEN, examples=EXTRA_EXAMPLES)
    time.sleep(3)

versions = sorted(client.list_dataset_versions(dataset_name=GOLDEN), key=lambda v: v.as_of)
print(f"{len(versions)} versions; tags:", [v.tags for v in versions])
print("examples at v1-baseline:", len(list(client.list_examples(dataset_name=GOLDEN, as_of="v1-baseline"))))
print("examples at latest     :", len(list(client.list_examples(dataset_name=GOLDEN))))

## 4. Splits and metadata

`list_examples` filters server-side by `splits` and by `metadata`.

In [ ]:
for split in ("train", "test"):
    print(split, len(list(client.list_examples(dataset_name=GOLDEN, splits=[split]))))
refunds = list(client.list_examples(dataset_name=GOLDEN, metadata={"segment": "refund"}))
print("segment=refund:", len(refunds))

## 5. Pull and push

To work outside LangSmith (CML, GCP, a notebook server), pull examples to a file. To bring edits back, push a file
as a new dataset. Here you pull the `test` split, add a case "locally", and push the result.

In [ ]:
os.makedirs(lab_sdk.EXPORT_DIR, exist_ok=True)
path = f"{lab_sdk.EXPORT_DIR}/golden_test.jsonl"

with open(path, "w") as f:
    for e in client.list_examples(dataset_name=GOLDEN, splits=["test"]):
        f.write(json.dumps({"inputs": e.inputs, "outputs": e.outputs, "metadata": e.metadata, "split": "test"}) + "\n")

with open(path) as f:
    rows = [json.loads(line) for line in f if line.strip()]
if not any(r["metadata"].get("origin") == "local-edit" for r in rows):
    rows.append({"inputs": {"query": "Do you ship internationally?"},
                 "outputs": {"trajectory": ["search_faq"], "keywords": ["shipping"]},
                 "metadata": {"segment": "faq", "origin": "local-edit"}, "split": "test"})
    with open(path, "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)

if client.has_dataset(dataset_name=UPLOADED):
    print(f"'{UPLOADED}' already exists; leaving it as is")
else:
    ds = client.create_dataset(UPLOADED, description="Pushed from a local JSONL file")
    client.create_examples(dataset_id=ds.id, examples=rows)
    print(f"Pushed {len(rows)} examples to '{UPLOADED}'")

## 6. See it in LangSmith

In [ ]:
lab_links.dataset_link(GOLDEN, hint="open Examples (filter by split), and the version history with the v1-baseline tag")
lab_links.dataset_link(UPLOADED, hint="the dataset you pushed from a local file")